# Module 13 — Logistic Regression for Fraud: An Interpretable Baseline

> **Track 2 · Revolut Fintech Specialization**  
> **Difficulty**: Intermediate  
> **Runtime**: ~4 minutes  
> **Key Libraries**: Scikit-learn, Statsmodels, NumPy, Pandas, Plotly, Seaborn  
> **Prerequisite**: Module 12 (Isolation Forest)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Fraud Dataset](#3-synthetic-fraud-dataset)
4. [Exploratory Data Analysis](#4-exploratory-data-analysis)
5. [Feature Engineering](#5-feature-engineering)
6. [Logistic Regression Training](#6-logistic-regression-training)
7. [Coefficient Analysis & Odds Ratios](#7-coefficient-analysis--odds-ratios)
8. [Threshold Selection: Youden's J](#8-threshold-selection-youdens-j)
9. [Business Cost Matrix](#9-business-cost-matrix)
10. [Model Evaluation](#10-model-evaluation)
11. [Visualization Dashboard](#11-visualization-dashboard)
12. [Key Business Takeaway](#12-key-business-takeaway)

---
## §1 · Business Context

### Why Logistic Regression at Revolut?

Despite the availability of complex models (XGBoost, neural networks),
**logistic regression remains the workhorse** of fraud detection at Revolut because:

| Advantage | Why It Matters |
|-----------|----------------|
| **Interpretable** | Regulators require explanations for every blocked transaction |
| **Fast** | < 1 ms inference — suitable for real-time scoring |
| **Calibrated** | Output is a probability (0–1), easy to threshold |
| **Robust** | Less prone to overfitting on small fraud datasets |
| **Baseline** | Every complex model must beat this benchmark |

**The compliance angle**: under EU PSD2 regulations, Revolut must explain **why**
a transaction was flagged. Logistic regression coefficients provide this directly:
- "Transaction blocked because amount is 5× user's average (coefficient: +2.3)"
- "Transaction blocked because merchant risk score is 95 (coefficient: +1.8)"

In this notebook we will:
1. Generate a 100K-transaction fraud dataset with 20 features.
2. Train a logistic regression baseline.
3. Interpret coefficients as **odds ratios** for business stakeholders.
4. Select the optimal threshold using **Youden's J statistic**.
5. Build a **business cost matrix** to quantify financial impact.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, accuracy_score, confusion_matrix,
    roc_curve, precision_recall_curve, classification_report
)

# ── Statsmodels (for detailed statistics) ────────────────────────
import statsmodels.api as sm

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 25)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Fraud Dataset

We simulate **100,000 transactions** with **20 features** and a **1.5% fraud rate**.

In [ ]:
# ── Configuration ────────────────────────────────────────────────
N_TXN = 100_000
FRAUD_RATE = 0.015
N_FRAUD = int(N_TXN * FRAUD_RATE)
N_LEGIT = N_TXN - N_FRAUD

print(f"Generating fraud dataset:")
print(f"  Transactions: {N_TXN:,}")
print(f"  Fraud rate   : {FRAUD_RATE*100:.1f}%")

np.random.seed(42)

# ── Feature names ────────────────────────────────────────────────
feature_names = [
    "amount", "amount_log", "hour", "is_weekend",
    "txn_count_1h", "txn_count_24h", "amount_sum_24h",
    "merchant_risk", "country_risk", "is_international",
    "distance_from_home", "velocity_1h", "unique_merchants_7d",
    "is_atm", "is_transfer", "user_tenure_days",
    "credit_score", "z_score_amount", "amount_to_income_ratio",
    "days_since_last_txn",
]

# ── Generate legitimate transactions ─────────────────────────────
X_legit = np.random.randn(N_LEGIT, len(feature_names))
X_legit[:, 0] = np.abs(np.random.lognormal(3.0, 1.0, N_LEGIT))
X_legit[:, 1] = np.log1p(X_legit[:, 0])
X_legit[:, 2] = np.random.randint(0, 24, N_LEGIT)
X_legit[:, 3] = np.random.binomial(1, 0.25, N_LEGIT)
X_legit[:, 4] = np.random.poisson(1, N_LEGIT)
X_legit[:, 5] = np.random.poisson(5, N_LEGIT)
X_legit[:, 7] = np.random.uniform(0, 50, N_LEGIT)
X_legit[:, 8] = np.random.uniform(0, 30, N_LEGIT)
X_legit[:, 9] = np.random.binomial(1, 0.15, N_LEGIT)
X_legit[:, 10] = np.random.exponential(50, N_LEGIT)
X_legit[:, 15] = np.random.exponential(365, N_LEGIT)
X_legit[:, 16] = np.random.normal(650, 80, N_LEGIT)
X_legit[:, 17] = np.random.normal(0, 1, N_LEGIT)

# ── Generate fraudulent transactions ─────────────────────────────
X_fraud = np.random.randn(N_FRAUD, len(feature_names))
X_fraud[:, 0] = np.abs(np.random.lognormal(5.0, 1.5, N_FRAUD))
X_fraud[:, 1] = np.log1p(X_fraud[:, 0])
X_fraud[:, 2] = np.random.choice([0, 1, 2, 3, 22, 23], N_FRAUD)
X_fraud[:, 4] = np.random.poisson(5, N_FRAUD)
X_fraud[:, 5] = np.random.poisson(15, N_FRAUD)
X_fraud[:, 7] = np.random.uniform(60, 100, N_FRAUD)
X_fraud[:, 8] = np.random.uniform(70, 100, N_FRAUD)
X_fraud[:, 9] = np.random.binomial(1, 0.60, N_FRAUD)
X_fraud[:, 10] = np.random.exponential(500, N_FRAUD)
X_fraud[:, 13] = np.random.binomial(1, 0.40, N_FRAUD)
X_fraud[:, 14] = np.random.binomial(1, 0.30, N_FRAUD)
X_fraud[:, 17] = np.random.normal(3, 1.5, N_FRAUD)

# ── Combine ──────────────────────────────────────────────────────
X = np.vstack([X_legit, X_fraud])
y = np.concatenate([np.zeros(N_LEGIT), np.ones(N_FRAUD)])

idx = np.random.permutation(N_TXN)
X = X[idx]
y = y[idx]

df = pd.DataFrame(X, columns=feature_names)
df["is_fraud"] = y

print(f"\n✓ Shape: {df.shape}")
print(f"  Fraud: {int(y.sum()):,} ({y.mean()*100:.2f}%)")
df.head()

---
## §4 · Exploratory Data Analysis

In [ ]:
# Key statistics by fraud status
key_features = ["amount", "merchant_risk", "country_risk", "z_score_amount", "distance_from_home"]
df.groupby("is_fraud")[key_features].agg(["mean", "median", "std"]).round(2)

In [ ]:
# Correlation with target
correlations = df[feature_names + ["is_fraud"]].corr()["is_fraud"].drop("is_fraud").sort_values(ascending=False)
fig = px.bar(
    x=correlations.values,
    y=correlations.index,
    orientation="h",
    title="Feature Correlation with Fraud (Top 20)",
    color=correlations.values,
    color_continuous_scale="RdBu_r",
)
fig.update_layout(height=500, yaxis=dict(autorange="reversed"))
fig.show()

---
## §5 · Feature Engineering

Add a few derived features that improve model performance.

In [ ]:
# Derived features
df["amount_squared"] = df["amount"] ** 2
df["risk_interaction"] = df["merchant_risk"] * df["country_risk"]
df["velocity_amount"] = df["velocity_1h"] * df["amount"]
df["tenure_risk"] = df["user_tenure_days"] * df["merchant_risk"]

# Update feature list
all_features = feature_names + ["amount_squared", "risk_interaction", "velocity_amount", "tenure_risk"]

print(f"Total features: {len(all_features)}")
print(f"  Original : {len(feature_names)}")
print(f"  Derived  : 4")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df[all_features], df["is_fraud"], test_size=0.3, stratify=df["is_fraud"], random_state=42
)

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTrain: {len(y_train):,} ({y_train.sum():.0f} fraud)")
print(f"Test : {len(y_test):,} ({y_test.sum():.0f} fraud)")

---
## §6 · Logistic Regression Training

In [ ]:
# Train logistic regression
lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",  # handle imbalanced data
    random_state=42,
    solver="lbfgs",
)

t0 = time.time()
lr.fit(X_train_scaled, y_train)
train_time = time.time() - t0

print(f"✓ Trained in {train_time:.4f}s")
print(f"  Intercept: {lr.intercept_[0]:.4f}")
print(f"  Coefficients: {len(lr.coef_[0])}")

# Predictions
y_pred = lr.predict(X_test_scaled)
y_prob = lr.predict_proba(X_test_scaled)[:, 1]

print(f"\nTest set predictions:")
print(f"  Predicted fraud: {y_pred.sum():,}")
print(f"  Actual fraud   : {y_test.sum():.0f}")

---
## §7 · Coefficient Analysis & Odds Ratios

### Interpreting Logistic Regression Coefficients

For a feature with coefficient β:
- **Odds Ratio** = e^β
- OR > 1: feature **increases** fraud probability
- OR < 1: feature **decreases** fraud probability
- OR = 1: no effect

In [ ]:
# Extract coefficients
coef_df = pd.DataFrame({
    "Feature": all_features,
    "Coefficient": lr.coef_[0],
    "Odds Ratio": np.exp(lr.coef_[0]),
    "Abs Coefficient": np.abs(lr.coef_[0]),
}).sort_values("Abs Coefficient", ascending=False)

print("=" * 80)
print("LOGISTIC REGRESSION COEFFICIENTS (sorted by magnitude)")
print("=" * 80)
print(coef_df.round(4).to_string(index=False))

print("\n💡 Interpretation:")
print("  - Odds Ratio > 1: feature increases fraud odds")
print("  - Odds Ratio < 1: feature decreases fraud odds")
print("  - Example: OR = 2.5 means 2.5× higher fraud odds per 1 std increase")

In [ ]:
# Visualize top 15 features by coefficient magnitude
fig = px.bar(
    coef_df.head(15),
    x="Coefficient",
    y="Feature",
    orientation="h",
    title="Top 15 Feature Coefficients (Logistic Regression)",
    color="Coefficient",
    color_continuous_scale="RdBu_r",
)
fig.update_layout(height=500, yaxis=dict(autorange="reversed"))
fig.show()

print("💡 Red = increases fraud probability")
print("   Blue = decreases fraud probability")

In [ ]:
# Business-friendly interpretation
print("=" * 80)
print("BUSINESS INTERPRETATION (for stakeholders)")
print("=" * 80)

for _, row in coef_df.head(10).iterrows():
    feature = row["Feature"]
    or_val = row["Odds Ratio"]
    
    if or_val > 1:
        direction = "increases"
        effect = f"{or_val:.2f}× higher"
    else:
        direction = "decreases"
        effect = f"{1/or_val:.2f}× lower"
    
    print(f"  • {feature:30s} {direction} fraud odds by {effect}")

---
## §8 · Threshold Selection: Youden's J Statistic

**Youden's J** = Sensitivity + Specificity − 1

The optimal threshold maximizes J, balancing true positives and true negatives.

In [ ]:
# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# Youden's J = TPR - FPR
youdens_j = tpr - fpr
optimal_idx = np.argmax(youdens_j)
optimal_threshold = thresholds[optimal_idx]
optimal_j = youdens_j[optimal_idx]

print("=" * 70)
print("THRESHOLD SELECTION: Youden's J Statistic")
print("=" * 70)
print(f"\nOptimal threshold: {optimal_threshold:.4f}")
print(f"Youden's J       : {optimal_j:.4f}")
print(f"  (J = 1.0 is perfect, J = 0.0 is random)")

print(f"\nAt this threshold:")
y_pred_optimal = (y_prob >= optimal_threshold).astype(int)
prec = precision_score(y_test, y_pred_optimal)
rec = recall_score(y_test, y_pred_optimal)
f1 = f1_score(y_test, y_pred_optimal)
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-Score : {f1:.4f}")

print(f"\n💡 Default threshold (0.5) is rarely optimal for imbalanced data")
print(f"   Youden's J balances sensitivity and specificity")

In [ ]:
# Visualize Youden's J
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=thresholds, y=tpr,
    mode="lines", name="Sensitivity (TPR)",
    line=dict(color="#00CC96", width=2.5),
))

fig.add_trace(go.Scatter(
    x=thresholds, y=1 - fpr,
    mode="lines", name="Specificity (1 - FPR)",
    line=dict(color="#636EFA", width=2.5),
))

fig.add_trace(go.Scatter(
    x=thresholds, y=youdens_j,
    mode="lines", name="Youden's J",
    line=dict(color="#EF553B", width=3),
))

# Mark optimal threshold
fig.add_vline(x=optimal_threshold, line_dash="dash", line_color="black",
              annotation_text=f"Optimal: {optimal_threshold:.3f}")

fig.update_layout(
    title="Threshold Selection: Sensitivity, Specificity, and Youden's J",
    xaxis_title="Decision Threshold",
    yaxis_title="Score",
    height=450,
    legend=dict(orientation="h", y=1.12),
)
fig.show()

---
## §9 · Business Cost Matrix

### Assigning Financial Costs to Confusion Matrix Cells

| | Predicted: Legit | Predicted: Fraud |
|---|---|---|
| **Actual: Legit** | £0 (correct) | £5 (review cost) |
| **Actual: Fraud** | £2,500 (missed fraud) | £0 (prevented) |

In [ ]:
# Define cost matrix
COST_FN = 2500  # £ lost per missed fraud (average fraud amount)
COST_FP = 5     # £ per manual review

def compute_business_cost(y_true, y_pred, cost_fn=COST_FN, cost_fp=COST_FP):
    """Compute total business cost of predictions."""
    cm = confusion_matrix(y_true, y_pred)
    fn = cm[1, 0]  # false negatives (missed fraud)
    fp = cm[0, 1]  # false positives (false alarms)
    
    total_cost = fn * cost_fn + fp * cost_fp
    return total_cost, fn, fp

# Compare different thresholds
thresholds_to_test = [0.1, 0.2, 0.3, optimal_threshold, 0.5, 0.7]
cost_results = []

for thresh in thresholds_to_test:
    y_pred_thresh = (y_prob >= thresh).astype(int)
    cost, fn, fp = compute_business_cost(y_test, y_pred_thresh)
    prec = precision_score(y_test, y_pred_thresh, zero_division=0)
    rec = recall_score(y_test, y_pred_thresh, zero_division=0)
    
    cost_results.append({
        "Threshold": thresh,
        "Precision": prec,
        "Recall": rec,
        "False Positives": fp,
        "False Negatives": fn,
        "Review Cost (£)": fp * COST_FP,
        "Fraud Loss (£)": fn * COST_FN,
        "Total Cost (£)": cost,
    })

df_cost = pd.DataFrame(cost_results).round(4)
print("=" * 90)
print("BUSINESS COST ANALYSIS (Test Set)")
print("=" * 90)
print(f"\nAssumptions:")
print(f"  Cost per missed fraud (FN): £{COST_FN:,}")
print(f"  Cost per false alarm (FP) : £{COST_FP}")
print(f"\n{df_cost.to_string(index=False)}")

best_threshold = df_cost.loc[df_cost["Total Cost (£)"].idxmin(), "Threshold"]
print(f"\n💡 Optimal threshold (min cost): {best_threshold:.3f}")

In [ ]:
# Visualize cost vs. threshold
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Cost Components vs. Threshold",
                                    "Total Cost vs. Threshold"])

fig.add_trace(go.Scatter(
    x=df_cost["Threshold"], y=df_cost["Fraud Loss (£)"],
    mode="lines+markers", name="Fraud Loss",
    line=dict(color="#EF553B", width=2.5),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_cost["Threshold"], y=df_cost["Review Cost (£)"],
    mode="lines+markers", name="Review Cost",
    line=dict(color="#636EFA", width=2.5),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_cost["Threshold"], y=df_cost["Total Cost (£)"],
    mode="lines+markers", name="Total Cost",
    line=dict(color="#FFA15A", width=3),
    marker=dict(size=10),
), row=1, col=2)

fig.add_vline(x=best_threshold, line_dash="dash", line_color="gray",
              annotation_text=f"Optimal: {best_threshold:.3f}", row=1, col=2)

fig.update_layout(height=420, legend=dict(orientation="h", y=1.12))
fig.update_xaxes(title_text="Threshold", row=1, col=1)
fig.update_xaxes(title_text="Threshold", row=1, col=2)
fig.show()

print("💡 Low threshold → catches more fraud but high review cost")
print("   High threshold → fewer reviews but misses more fraud")
print("   Optimal balances the two costs")

---
## §10 · Model Evaluation

In [ ]:
# Final evaluation at optimal threshold
y_pred_final = (y_prob >= optimal_threshold).astype(int)

print("=" * 70)
print("FINAL MODEL EVALUATION (Optimal Threshold)")
print("=" * 70)
print(f"\nAccuracy : {accuracy_score(y_test, y_pred_final):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_final):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_final):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test, y_prob):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_final))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_final)
print("Confusion Matrix:")
print(cm)

---
## §11 · Visualization Dashboard

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr, mode="lines",
    name=f"Logistic Regression (AUC = {auc_score:.4f})",
    line=dict(color="#636EFA", width=2.5),
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines",
    name="Random Classifier",
    line=dict(color="gray", width=1, dash="dash"),
))

fig.update_layout(
    title="ROC Curve",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=450,
)
fig.show()

In [ ]:
# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=recall, y=precision, mode="lines",
    name=f"PR Curve (AUC = {pr_auc:.4f})",
    line=dict(color="#00CC96", width=2.5),
))

fig.update_layout(
    title="Precision-Recall Curve",
    xaxis_title="Recall",
    yaxis_title="Precision",
    height=450,
)
fig.show()

In [ ]:
# Business Impact Summary
total_cost, fn, fp = compute_business_cost(y_test, y_pred_final)
fraud_prevented = cm[1, 1]
avg_fraud_amount = COST_FN

print("=" * 70)
print("BUSINESS IMPACT SUMMARY (Monthly, 100K transactions)")
print("=" * 70)
print(f"\nFraud prevented      : {fraud_prevented:,} cases")
print(f"Fraud missed         : {fn:,} cases")
print(f"False alarms         : {fp:,} cases")
print(f"\nValue saved (fraud prevented) : £{fraud_prevented * avg_fraud_amount:,.0f}")
print(f"Cost of missed fraud        : £{fn * avg_fraud_amount:,.0f}")
print(f"Cost of manual reviews      : £{fp * COST_FP:,.0f}")
print(f"Total cost                  : £{total_cost:,.0f}")
print(f"Net value                   : £{fraud_prevented * avg_fraud_amount - total_cost:,.0f}")

---
## §12 · Key Business Takeaway

> ### 🎯 Action Items for a Fintech Data Scientist
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Baseline model, compliance-heavy domains, small datasets |
> | **Interpretability** | Coefficients → odds ratios → business explanations |
> | **Threshold** | Use Youden's J or business cost matrix (never default 0.5) |
> | **Imbalanced data** | Always use `class_weight='balanced'` |
> | **Scaling** | Standardize features before training |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Logistic regression is the compliance-friendly baseline**:
>    - Every blocked transaction can be explained via coefficients.
>    - Regulators accept logistic regression as "explainable AI".
>
> 2. **Odds ratios are business-friendly**:
>    ```
>    "Your transaction was flagged because:"
>    "  • Amount is 5× your average (risk factor: 2.3×)"
>    "  • Merchant is in a high-risk category (risk factor: 1.8×)"
>    "  • Transaction is international (risk factor: 1.5×)"
>    ```
>
> 3. **Always compare complex models against this baseline**:
>    - If XGBoost improves AUC by < 2%, stick with logistic regression.
>    - Complexity must justify the marginal gain.
>
> 4. **Threshold selection is a business decision**, not just statistical:
>    - Low threshold: catch more fraud, higher review cost.
>    - High threshold: fewer reviews, more missed fraud.
>    - Use the cost matrix to find the optimal balance.
>
> ### 📌 Logistic Regression Cheat Sheet
>
> ```python
> from sklearn.linear_model import LogisticRegression
> from sklearn.preprocessing import StandardScaler
>
> # Always scale features
> scaler = StandardScaler()
> X_train_scaled = scaler.fit_transform(X_train)
>
> # Train with balanced weights
> lr = LogisticRegression(
>     class_weight="balanced",  # handle imbalanced data
>     max_iter=1000,
>     random_state=42,
> )
> lr.fit(X_train_scaled, y_train)
>
> # Interpret coefficients
> odds_ratios = np.exp(lr.coef_[0])
> # OR > 1: increases fraud probability
> # OR < 1: decreases fraud probability
> ```
>
> ### 🔑 Key Takeaways
>
> 1. **Logistic regression is interpretable** — essential for compliance and customer disputes.
> 2. **Odds ratios translate coefficients into business language**.
> 3. **Youden's J** or **business cost matrix** should guide threshold selection.
> 4. **Always use `class_weight='balanced'`** for imbalanced fraud data.
> 5. **This baseline must be beaten** by any complex model to justify deployment.